In [1]:
#!/usr/bin/env python3
"""
JAX Adam PINN for the reduced MOOSE neutron--thermal benchmark.

This script is intentionally clean and configurable:
  - Adam optimizer implemented in this file. No optax dependency.
  - Fresh collocation/minibatch points are resampled every Adam step.
  - Boundary/interface points are sampled exactly on the boundaries: no jitter/noise.
  - Learning rate is configurable and can use constant or cosine schedule.
  - Loss-weight group schedules are configurable.
  - Point counts are configurable per PDE/BC/IC/interface component.

MOOSE model represented here:
  Fuel domain:    x in [0, 0.4], y in [0, 1], t in [0, 1]
  Coolant domain: x in [0.4, 1], y in [0, 1], t in [0, 1]

Unknowns:
  phi(x,y,t): neutron flux in fuel only
  Ts(x,y,t): fuel temperature
  Tf(x,y,t): coolant temperature

PDEs:
  phi_t - D_phi*(phi_xx + phi_yy) + Sigma_a*phi = 0              in fuel
  rho_s_cp_s*Ts_t - k_s*(Ts_xx + Ts_yy) - gamma*phi = 0          in fuel
  rho_f_cp_f*Tf_t + vy*Tf_y - k_f*(Tf_xx + Tf_yy) = 0            in coolant

BC/IC/interface:
  phi(0,y,t) = (1-exp(-5t))*(1 + 0.2*sin(pi*y)^2)
  phi_y=0 on fuel bottom/top, phi_x=0 on interface
  Ts_x=0 on fuel left, Ts_y=0 on fuel bottom/top
  Tf=0 on coolant bottom, Tf_y=0 on coolant top, Tf_x=0 on coolant right
  phi=Ts=Tf=0 initial conditions on their blocks at t=0
  Ts=Tf and -k_s*Ts_x + k_f*Tf_x=0 at x=0.4 interface

Examples:
  # Small CPU smoke run
  python adam_jax_moose_resample.py --steps 10 --device cpu --jit

  # Real run with default point counts and cosine LR
  python adam_jax_moose_resample.py --steps 10000 --device gpu --lr 1e-3 --lr-final 2e-4

  # More points on GPU
  python adam_jax_moose_resample.py \
      --steps 20000 --device gpu --lr 1e-3 \
      --n-phi-pde 2048 --n-ts-pde 2048 --n-tf-pde 4096 \
      --n-interface 2048
"""

from __future__ import annotations

import argparse
import csv
import math
import os
import pickle
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Dict, Tuple

# Keep JAX memory behavior less aggressive on shared GPUs.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import jax
import jax.numpy as jnp
from jax import random
from jax.tree_util import tree_leaves, tree_map


# =============================================================================
# Configuration
# =============================================================================

@dataclass
class Config:
    # Domain
    x_fuel_min: float = 0.0
    x_fuel_max: float = 0.4
    x_cool_min: float = 0.4
    x_cool_max: float = 1.0
    y_min: float = 0.0
    y_max: float = 1.0
    t_min: float = 0.0
    t_max: float = 1.0

    # Physics from the MOOSE input
    D_phi: float = 0.05
    Sigma_a: float = 1.0
    rho_s_cp_s: float = 1.0
    rho_f_cp_f: float = 1.0
    k_s: float = 0.50
    k_f: float = 0.15
    gamma: float = 10.0
    vy: float = 1.0

    # Network
    width: int = 64
    layers: int = 4
    activation: str = "tanh"  # tanh or silu
    phi_scale: float = 1.5
    T_scale: float = 3.0

    # Adam / training
    steps: int = 50000
    lr: float = 1.0e-3
    lr_final: float = 2.0e-4
    lr_schedule: str = "cosine"  # constant or cosine
    lr_warmup_steps: int = 0
    beta1: float = 0.9
    beta2: float = 0.999
    adam_eps: float = 1.0e-8
    grad_clip_norm: float = 10.0
    seed: int = 0
    dtype: str = "float32"  # float32 or float64
    device: str = "auto"    # auto, cpu, gpu, cuda, cuda:0, gpu:0
    use_jit: bool = True

    # Fresh points per Adam step. These are the minibatch sizes.
    # PDE points
    n_phi_pde: int = 512
    n_ts_pde: int = 512
    n_tf_pde: int = 768

    # phi BC points
    n_phi_left: int = 128
    n_phi_bottom: int = 128
    n_phi_top: int = 128
    n_phi_interface_neumann: int = 128

    # Ts BC points
    n_ts_left: int = 128
    n_ts_bottom: int = 128
    n_ts_top: int = 128

    # Tf BC points
    n_tf_bottom: int = 128
    n_tf_top: int = 128
    n_tf_right: int = 128

    # IC points
    n_ic_phi: int = 128
    n_ic_ts: int = 128
    n_ic_tf: int = 128

    # Interface points
    n_interface: int = 256

    # Component base weights
    w_phi_pde: float = 1.0
    w_ts_pde: float = 1.0
    w_tf_pde: float = 3.0

    w_phi_left: float = 25.0
    w_phi_neumann: float = 5.0
    w_ts_neumann: float = 5.0
    w_tf_bottom: float = 25.0
    w_tf_neumann: float = 5.0

    w_ic_phi: float = 10.0
    w_ic_ts: float = 10.0
    w_ic_tf: float = 10.0

    w_interface_T: float = 50.0
    w_interface_flux: float = 20.0

    # Group multiplier schedule for loss weights.
    # Final weighted loss = pde_mult*(PDE losses) + bc_mult*(BC losses)
    #                     + ic_mult*(IC losses) + interface_mult*(interface losses)
    weight_schedule: str = "linear"  # none, linear, cosine
    weight_warmup_steps: int = 2000
    pde_mult_start: float = 0.2
    pde_mult_end: float = 1.0
    bc_mult_start: float = 1.0
    bc_mult_end: float = 1.0
    ic_mult_start: float = 1.0
    ic_mult_end: float = 0.5
    interface_mult_start: float = 0.2
    interface_mult_end: float = 1.0

    # Output
    out_dir: str = "adam_jax_moose_output"
    log_every: int = 100
    save_every: int = 2000
    export_grid_nx: int = 161
    export_grid_ny: int = 161
    export_time: float = 1.0


# =============================================================================
# Basic utilities
# =============================================================================

def dtype_from_config(cfg: Config):
    if cfg.dtype == "float32":
        return jnp.float32
    if cfg.dtype == "float64":
        jax.config.update("jax_enable_x64", True)
        return jnp.float64
    raise ValueError("dtype must be float32 or float64")


def choose_device(device_arg: str):
    arg = device_arg.lower()
    if arg == "auto":
        return jax.devices()[0]
    if arg == "cpu":
        return jax.devices("cpu")[0]
    if arg in ("gpu", "cuda"):
        return jax.devices("gpu")[0]
    if arg.startswith("cuda:") or arg.startswith("gpu:"):
        idx = int(arg.split(":", 1)[1])
        return jax.devices("gpu")[idx]
    raise ValueError("device must be auto, cpu, gpu, cuda, cuda:0, gpu:0, ...")


def safe_float(x: Any) -> float:
    return float(jax.device_get(x))


def tree_l2_norm(tree) -> jnp.ndarray:
    leaves = tree_leaves(tree)
    if not leaves:
        return jnp.asarray(0.0)
    total = sum(jnp.sum(jnp.square(x)) for x in leaves)
    return jnp.sqrt(total)


def clip_grads(grads, max_norm: float):
    norm = tree_l2_norm(grads)
    if max_norm is None or max_norm <= 0:
        return grads, norm
    scale = jnp.minimum(1.0, max_norm / (norm + 1.0e-12))
    return tree_map(lambda g: scale * g, grads), norm


# =============================================================================
# MLP model
# =============================================================================

def init_mlp(key, in_dim: int, out_dim: int, width: int, depth: int, dtype):
    keys = random.split(key, depth + 1)
    params = []
    last_dim = in_dim
    for i in range(depth):
        k = keys[i]
        std = math.sqrt(2.0 / float(last_dim + width))
        W = std * random.normal(k, (last_dim, width), dtype=dtype)
        b = jnp.zeros((width,), dtype=dtype)
        params.append((W, b))
        last_dim = width

    std = math.sqrt(2.0 / float(last_dim + out_dim))
    W = std * random.normal(keys[-1], (last_dim, out_dim), dtype=dtype)
    b = jnp.zeros((out_dim,), dtype=dtype)
    params.append((W, b))
    return params


def init_params(key, cfg: Config, dtype):
    k_phi, k_ts, k_tf = random.split(key, 3)
    return {
        "phi": init_mlp(k_phi, 3, 1, cfg.width, cfg.layers, dtype),
        "ts": init_mlp(k_ts, 3, 1, cfg.width, cfg.layers, dtype),
        "tf": init_mlp(k_tf, 3, 1, cfg.width, cfg.layers, dtype),
    }


def activate(z: jnp.ndarray, activation: str) -> jnp.ndarray:
    if activation == "tanh":
        return jnp.tanh(z)
    if activation == "silu":
        return z * jax.nn.sigmoid(z)
    raise ValueError("activation must be tanh or silu")


def mlp_apply(mlp_params, x: jnp.ndarray, activation: str) -> jnp.ndarray:
    h = x
    for W, b in mlp_params[:-1]:
        h = activate(jnp.dot(h, W) + b, activation)
    W, b = mlp_params[-1]
    return jnp.dot(h, W) + b


def normalize_xyt(xyt: jnp.ndarray) -> jnp.ndarray:
    # The full domain is [0,1]^3, so this maps to [-1,1]^3.
    return 2.0 * xyt - 1.0


def forward_all(params, xyt: jnp.ndarray, cfg: Config) -> jnp.ndarray:
    z = normalize_xyt(xyt)
    phi = cfg.phi_scale * mlp_apply(params["phi"], z, cfg.activation)
    ts = cfg.T_scale * mlp_apply(params["ts"], z, cfg.activation)
    tf = cfg.T_scale * mlp_apply(params["tf"], z, cfg.activation)
    return jnp.concatenate([phi, ts, tf], axis=-1)


def phi_scalar(params, xyt: jnp.ndarray, cfg: Config) -> jnp.ndarray:
    return forward_all(params, xyt, cfg)[0]


def ts_scalar(params, xyt: jnp.ndarray, cfg: Config) -> jnp.ndarray:
    return forward_all(params, xyt, cfg)[1]


def tf_scalar(params, xyt: jnp.ndarray, cfg: Config) -> jnp.ndarray:
    return forward_all(params, xyt, cfg)[2]


# =============================================================================
# Sampling. Boundary coordinates are exact; no jitter/noise is added.
# =============================================================================

def rand_col(key, n: int, low: float, high: float, dtype):
    return low + (high - low) * random.uniform(key, (n, 1), dtype=dtype)


def sample_box(key, n: int, xmin: float, xmax: float, ymin: float, ymax: float,
               tmin: float, tmax: float, dtype) -> jnp.ndarray:
    kx, ky, kt = random.split(key, 3)
    x = rand_col(kx, n, xmin, xmax, dtype)
    y = rand_col(ky, n, ymin, ymax, dtype)
    t = rand_col(kt, n, tmin, tmax, dtype)
    return jnp.concatenate([x, y, t], axis=1)


def sample_fuel(key, n: int, cfg: Config, dtype):
    return sample_box(key, n, cfg.x_fuel_min, cfg.x_fuel_max, cfg.y_min, cfg.y_max,
                      cfg.t_min, cfg.t_max, dtype)


def sample_coolant(key, n: int, cfg: Config, dtype):
    return sample_box(key, n, cfg.x_cool_min, cfg.x_cool_max, cfg.y_min, cfg.y_max,
                      cfg.t_min, cfg.t_max, dtype)


def sample_ic_fuel(key, n: int, cfg: Config, dtype):
    kx, ky = random.split(key, 2)
    x = rand_col(kx, n, cfg.x_fuel_min, cfg.x_fuel_max, dtype)
    y = rand_col(ky, n, cfg.y_min, cfg.y_max, dtype)
    t = jnp.full((n, 1), cfg.t_min, dtype=dtype)
    return jnp.concatenate([x, y, t], axis=1)


def sample_ic_coolant(key, n: int, cfg: Config, dtype):
    kx, ky = random.split(key, 2)
    x = rand_col(kx, n, cfg.x_cool_min, cfg.x_cool_max, dtype)
    y = rand_col(ky, n, cfg.y_min, cfg.y_max, dtype)
    t = jnp.full((n, 1), cfg.t_min, dtype=dtype)
    return jnp.concatenate([x, y, t], axis=1)


def sample_boundary(key, n: int, cfg: Config, dtype, block: str, side: str) -> jnp.ndarray:
    if block == "fuel":
        xmin, xmax = cfg.x_fuel_min, cfg.x_fuel_max
    elif block == "coolant":
        xmin, xmax = cfg.x_cool_min, cfg.x_cool_max
    else:
        raise ValueError("block must be fuel or coolant")

    k1, k2 = random.split(key, 2)
    if side == "left":
        x = jnp.full((n, 1), xmin, dtype=dtype)
        y = rand_col(k1, n, cfg.y_min, cfg.y_max, dtype)
    elif side == "right":
        x = jnp.full((n, 1), xmax, dtype=dtype)
        y = rand_col(k1, n, cfg.y_min, cfg.y_max, dtype)
    elif side == "interface":
        x = jnp.full((n, 1), cfg.x_fuel_max, dtype=dtype)
        y = rand_col(k1, n, cfg.y_min, cfg.y_max, dtype)
    elif side == "bottom":
        x = rand_col(k1, n, xmin, xmax, dtype)
        y = jnp.full((n, 1), cfg.y_min, dtype=dtype)
    elif side == "top":
        x = rand_col(k1, n, xmin, xmax, dtype)
        y = jnp.full((n, 1), cfg.y_max, dtype=dtype)
    else:
        raise ValueError("side must be left, right, interface, bottom, or top")

    t = rand_col(k2, n, cfg.t_min, cfg.t_max, dtype)
    return jnp.concatenate([x, y, t], axis=1)


def sample_interface(key, n: int, cfg: Config, dtype) -> jnp.ndarray:
    ky, kt = random.split(key, 2)
    x = jnp.full((n, 1), cfg.x_fuel_max, dtype=dtype)
    y = rand_col(ky, n, cfg.y_min, cfg.y_max, dtype)
    t = rand_col(kt, n, cfg.t_min, cfg.t_max, dtype)
    return jnp.concatenate([x, y, t], axis=1)


# =============================================================================
# Derivatives and residuals
# =============================================================================

def grad_xyt(fn: Callable[[jnp.ndarray], jnp.ndarray], xyt: jnp.ndarray) -> jnp.ndarray:
    return jax.grad(fn)(xyt)


def hess_xyt(fn: Callable[[jnp.ndarray], jnp.ndarray], xyt: jnp.ndarray) -> jnp.ndarray:
    return jax.hessian(fn)(xyt)


def phi_residual_single(params, xyt: jnp.ndarray, cfg: Config) -> jnp.ndarray:
    fn = lambda z: phi_scalar(params, z, cfg)
    g = grad_xyt(fn, xyt)
    H = hess_xyt(fn, xyt)
    lap = H[0, 0] + H[1, 1]
    return g[2] - cfg.D_phi * lap + cfg.Sigma_a * fn(xyt)


def ts_residual_single(params, xyt: jnp.ndarray, cfg: Config) -> jnp.ndarray:
    fn = lambda z: ts_scalar(params, z, cfg)
    g = grad_xyt(fn, xyt)
    H = hess_xyt(fn, xyt)
    lap = H[0, 0] + H[1, 1]
    phi_val = phi_scalar(params, xyt, cfg)
    return cfg.rho_s_cp_s * g[2] - cfg.k_s * lap - cfg.gamma * phi_val


def tf_residual_single(params, xyt: jnp.ndarray, cfg: Config) -> jnp.ndarray:
    fn = lambda z: tf_scalar(params, z, cfg)
    g = grad_xyt(fn, xyt)
    H = hess_xyt(fn, xyt)
    lap = H[0, 0] + H[1, 1]
    return cfg.rho_f_cp_f * g[2] + cfg.vy * g[1] - cfg.k_f * lap


def mse(x: jnp.ndarray) -> jnp.ndarray:
    return jnp.mean(jnp.square(x))


def phi_left_target(xyt: jnp.ndarray) -> jnp.ndarray:
    y = xyt[:, 1]
    t = xyt[:, 2]
    return (1.0 - jnp.exp(-5.0 * t)) * (1.0 + 0.2 * jnp.sin(jnp.pi * y) ** 2)


def derivative_component(params, xyt_batch: jnp.ndarray, cfg: Config,
                         field: str, component: int) -> jnp.ndarray:
    if field == "phi":
        fn = lambda z: phi_scalar(params, z, cfg)
    elif field == "ts":
        fn = lambda z: ts_scalar(params, z, cfg)
    elif field == "tf":
        fn = lambda z: tf_scalar(params, z, cfg)
    else:
        raise ValueError("field must be phi, ts, or tf")
    return jax.vmap(lambda z: jax.grad(fn)(z)[component])(xyt_batch)


# =============================================================================
# Loss calculation
# =============================================================================

def schedule_multiplier(step: jnp.ndarray, start: float, end: float, warmup_steps: int, kind: str, dtype):
    if kind == "none" or warmup_steps <= 0:
        return jnp.asarray(end, dtype=dtype)
    s = jnp.asarray(step, dtype=dtype)
    p = jnp.clip(s / float(warmup_steps), 0.0, 1.0)
    if kind == "linear":
        a = p
    elif kind == "cosine":
        a = 0.5 - 0.5 * jnp.cos(jnp.pi * p)
    else:
        raise ValueError("weight_schedule must be none, linear, or cosine")
    return jnp.asarray(start, dtype=dtype) + (jnp.asarray(end, dtype=dtype) - jnp.asarray(start, dtype=dtype)) * a


def learning_rate_at_step(step: jnp.ndarray, cfg: Config, dtype) -> jnp.ndarray:
    s = jnp.asarray(step, dtype=dtype)
    if cfg.lr_schedule == "constant":
        body_lr = jnp.asarray(cfg.lr, dtype=dtype)
    elif cfg.lr_schedule == "cosine":
        denom = max(1, cfg.steps - cfg.lr_warmup_steps)
        p = jnp.clip((s - float(cfg.lr_warmup_steps)) / float(denom), 0.0, 1.0)
        body_lr = cfg.lr_final + 0.5 * (cfg.lr - cfg.lr_final) * (1.0 + jnp.cos(jnp.pi * p))
        body_lr = jnp.asarray(body_lr, dtype=dtype)
    else:
        raise ValueError("lr_schedule must be constant or cosine")

    if cfg.lr_warmup_steps > 0:
        warm = jnp.clip(s / float(cfg.lr_warmup_steps), 0.0, 1.0)
        return body_lr * warm
    return body_lr


def make_total_loss(cfg: Config, dtype):
    """Return total_loss(params, key, step) as a closure over cfg."""

    def total_loss(params, key, step):
        # Split once per Adam step. Every step gets a new key, so every component is freshly resampled.
        keys = random.split(key, 24)

        # PDE losses
        x_phi = sample_fuel(keys[0], cfg.n_phi_pde, cfg, dtype)
        r_phi = jax.vmap(lambda z: phi_residual_single(params, z, cfg))(x_phi)
        l_phi_pde = mse(r_phi)

        x_ts = sample_fuel(keys[1], cfg.n_ts_pde, cfg, dtype)
        r_ts = jax.vmap(lambda z: ts_residual_single(params, z, cfg))(x_ts)
        l_ts_pde = mse(r_ts)

        x_tf = sample_coolant(keys[2], cfg.n_tf_pde, cfg, dtype)
        r_tf = jax.vmap(lambda z: tf_residual_single(params, z, cfg))(x_tf)
        l_tf_pde = mse(r_tf)

        # phi BC losses
        x_phi_left = sample_boundary(keys[3], cfg.n_phi_left, cfg, dtype, "fuel", "left")
        phi_left_pred = jax.vmap(lambda z: phi_scalar(params, z, cfg))(x_phi_left)
        l_phi_left = mse(phi_left_pred - phi_left_target(x_phi_left))

        x_phi_bottom = sample_boundary(keys[4], cfg.n_phi_bottom, cfg, dtype, "fuel", "bottom")
        x_phi_top = sample_boundary(keys[5], cfg.n_phi_top, cfg, dtype, "fuel", "top")
        x_phi_if = sample_boundary(keys[6], cfg.n_phi_interface_neumann, cfg, dtype, "fuel", "interface")
        phi_bottom_y = derivative_component(params, x_phi_bottom, cfg, "phi", 1)
        phi_top_y = derivative_component(params, x_phi_top, cfg, "phi", 1)
        phi_if_x = derivative_component(params, x_phi_if, cfg, "phi", 0)
        l_phi_neumann = mse(phi_bottom_y) + mse(phi_top_y) + mse(phi_if_x)

        # Ts BC losses
        x_ts_left = sample_boundary(keys[7], cfg.n_ts_left, cfg, dtype, "fuel", "left")
        x_ts_bottom = sample_boundary(keys[8], cfg.n_ts_bottom, cfg, dtype, "fuel", "bottom")
        x_ts_top = sample_boundary(keys[9], cfg.n_ts_top, cfg, dtype, "fuel", "top")
        ts_left_x = derivative_component(params, x_ts_left, cfg, "ts", 0)
        ts_bottom_y = derivative_component(params, x_ts_bottom, cfg, "ts", 1)
        ts_top_y = derivative_component(params, x_ts_top, cfg, "ts", 1)
        l_ts_neumann = mse(ts_left_x) + mse(ts_bottom_y) + mse(ts_top_y)

        # Tf BC losses
        x_tf_bottom = sample_boundary(keys[10], cfg.n_tf_bottom, cfg, dtype, "coolant", "bottom")
        tf_bottom_pred = jax.vmap(lambda z: tf_scalar(params, z, cfg))(x_tf_bottom)
        l_tf_bottom = mse(tf_bottom_pred)

        x_tf_top = sample_boundary(keys[11], cfg.n_tf_top, cfg, dtype, "coolant", "top")
        x_tf_right = sample_boundary(keys[12], cfg.n_tf_right, cfg, dtype, "coolant", "right")
        tf_top_y = derivative_component(params, x_tf_top, cfg, "tf", 1)
        tf_right_x = derivative_component(params, x_tf_right, cfg, "tf", 0)
        l_tf_neumann = mse(tf_top_y) + mse(tf_right_x)

        # IC losses
        x_ic_phi = sample_ic_fuel(keys[13], cfg.n_ic_phi, cfg, dtype)
        x_ic_ts = sample_ic_fuel(keys[14], cfg.n_ic_ts, cfg, dtype)
        x_ic_tf = sample_ic_coolant(keys[15], cfg.n_ic_tf, cfg, dtype)
        l_ic_phi = mse(jax.vmap(lambda z: phi_scalar(params, z, cfg))(x_ic_phi))
        l_ic_ts = mse(jax.vmap(lambda z: ts_scalar(params, z, cfg))(x_ic_ts))
        l_ic_tf = mse(jax.vmap(lambda z: tf_scalar(params, z, cfg))(x_ic_tf))

        # Interface losses
        x_int = sample_interface(keys[16], cfg.n_interface, cfg, dtype)
        ts_int = jax.vmap(lambda z: ts_scalar(params, z, cfg))(x_int)
        tf_int = jax.vmap(lambda z: tf_scalar(params, z, cfg))(x_int)
        l_interface_T = mse(ts_int - tf_int)
        ts_x = derivative_component(params, x_int, cfg, "ts", 0)
        tf_x = derivative_component(params, x_int, cfg, "tf", 0)
        l_interface_flux = mse(-cfg.k_s * ts_x + cfg.k_f * tf_x)

        # Group schedules
        pde_mult = schedule_multiplier(step, cfg.pde_mult_start, cfg.pde_mult_end,
                                       cfg.weight_warmup_steps, cfg.weight_schedule, dtype)
        bc_mult = schedule_multiplier(step, cfg.bc_mult_start, cfg.bc_mult_end,
                                      cfg.weight_warmup_steps, cfg.weight_schedule, dtype)
        ic_mult = schedule_multiplier(step, cfg.ic_mult_start, cfg.ic_mult_end,
                                      cfg.weight_warmup_steps, cfg.weight_schedule, dtype)
        interface_mult = schedule_multiplier(step, cfg.interface_mult_start, cfg.interface_mult_end,
                                             cfg.weight_warmup_steps, cfg.weight_schedule, dtype)

        pde_loss = (
            cfg.w_phi_pde * l_phi_pde
            + cfg.w_ts_pde * l_ts_pde
            + cfg.w_tf_pde * l_tf_pde
        )
        bc_loss = (
            cfg.w_phi_left * l_phi_left
            + cfg.w_phi_neumann * l_phi_neumann
            + cfg.w_ts_neumann * l_ts_neumann
            + cfg.w_tf_bottom * l_tf_bottom
            + cfg.w_tf_neumann * l_tf_neumann
        )
        ic_loss = cfg.w_ic_phi * l_ic_phi + cfg.w_ic_ts * l_ic_ts + cfg.w_ic_tf * l_ic_tf
        interface_loss = cfg.w_interface_T * l_interface_T + cfg.w_interface_flux * l_interface_flux

        total = pde_mult * pde_loss + bc_mult * bc_loss + ic_mult * ic_loss + interface_mult * interface_loss

        logs = {
            "total": total,
            "phi_pde": l_phi_pde,
            "ts_pde": l_ts_pde,
            "tf_pde": l_tf_pde,
            "phi_left": l_phi_left,
            "phi_neumann": l_phi_neumann,
            "ts_neumann": l_ts_neumann,
            "tf_bottom": l_tf_bottom,
            "tf_neumann": l_tf_neumann,
            "ic_phi": l_ic_phi,
            "ic_ts": l_ic_ts,
            "ic_tf": l_ic_tf,
            "interface_T": l_interface_T,
            "interface_flux": l_interface_flux,
            "pde_mult": pde_mult,
            "bc_mult": bc_mult,
            "ic_mult": ic_mult,
            "interface_mult": interface_mult,
        }
        return total, logs

    return total_loss


# =============================================================================
# Adam optimizer
# =============================================================================

def init_adam_state(params, dtype):
    zeros = tree_map(lambda p: jnp.zeros_like(p, dtype=dtype), params)
    return {
        "m": zeros,
        "v": zeros,
        "t": jnp.asarray(0, dtype=jnp.int32),
    }


def adam_update(params, opt_state, grads, lr: jnp.ndarray, cfg: Config):
    t = opt_state["t"] + jnp.asarray(1, dtype=jnp.int32)
    beta1 = jnp.asarray(cfg.beta1)
    beta2 = jnp.asarray(cfg.beta2)

    m = tree_map(lambda m_old, g: beta1 * m_old + (1.0 - beta1) * g, opt_state["m"], grads)
    v = tree_map(lambda v_old, g: beta2 * v_old + (1.0 - beta2) * jnp.square(g), opt_state["v"], grads)

    one = jnp.asarray(1.0)
    b1_corr = one - beta1 ** t
    b2_corr = one - beta2 ** t

    m_hat = tree_map(lambda x: x / b1_corr, m)
    v_hat = tree_map(lambda x: x / b2_corr, v)

    new_params = tree_map(
        lambda p, mh, vh: p - lr * mh / (jnp.sqrt(vh) + cfg.adam_eps),
        params,
        m_hat,
        v_hat,
    )
    return new_params, {"m": m, "v": v, "t": t}


def make_train_step(cfg: Config, dtype):
    total_loss = make_total_loss(cfg, dtype)

    def train_step(params, opt_state, key, step):
        (loss_value, logs), grads = jax.value_and_grad(total_loss, has_aux=True)(params, key, step)
        grads, grad_norm = clip_grads(grads, cfg.grad_clip_norm)
        lr = learning_rate_at_step(step, cfg, dtype)
        params, opt_state = adam_update(params, opt_state, grads, lr, cfg)
        logs = dict(logs)
        logs["total"] = loss_value
        logs["grad_norm"] = grad_norm
        logs["lr"] = lr
        return params, opt_state, logs

    if cfg.use_jit:
        return jax.jit(train_step)
    return train_step


# =============================================================================
# Output helpers
# =============================================================================

def write_config(cfg: Config, out_dir: Path) -> None:
    path = out_dir / "config.txt"
    with path.open("w") as f:
        for k, v in asdict(cfg).items():
            f.write(f"{k} = {v}\n")


def write_loss_header(path: Path) -> None:
    columns = [
        "step", "time_sec", "lr", "total", "grad_norm",
        "phi_pde", "ts_pde", "tf_pde",
        "phi_left", "phi_neumann", "ts_neumann", "tf_bottom", "tf_neumann",
        "ic_phi", "ic_ts", "ic_tf",
        "interface_T", "interface_flux",
        "pde_mult", "bc_mult", "ic_mult", "interface_mult",
    ]
    with path.open("w", newline="") as f:
        csv.writer(f).writerow(columns)


def append_loss_row(path: Path, step: int, elapsed: float, logs: Dict[str, Any]) -> None:
    row = [
        step,
        elapsed,
        safe_float(logs["lr"]),
        safe_float(logs["total"]),
        safe_float(logs["grad_norm"]),
        safe_float(logs["phi_pde"]),
        safe_float(logs["ts_pde"]),
        safe_float(logs["tf_pde"]),
        safe_float(logs["phi_left"]),
        safe_float(logs["phi_neumann"]),
        safe_float(logs["ts_neumann"]),
        safe_float(logs["tf_bottom"]),
        safe_float(logs["tf_neumann"]),
        safe_float(logs["ic_phi"]),
        safe_float(logs["ic_ts"]),
        safe_float(logs["ic_tf"]),
        safe_float(logs["interface_T"]),
        safe_float(logs["interface_flux"]),
        safe_float(logs["pde_mult"]),
        safe_float(logs["bc_mult"]),
        safe_float(logs["ic_mult"]),
        safe_float(logs["interface_mult"]),
    ]
    with path.open("a", newline="") as f:
        csv.writer(f).writerow(row)


def save_checkpoint(params, opt_state, cfg: Config, step: int, out_dir: Path) -> None:
    path_latest = out_dir / "checkpoint_latest.pkl"
    payload = {
        "step": step,
        "config": asdict(cfg),
        "params": jax.device_get(params),
        "opt_state": jax.device_get(opt_state),
    }
    with path_latest.open("wb") as f:
        pickle.dump(payload, f)
    path_step = out_dir / f"checkpoint_step_{step}.pkl"
    with path_step.open("wb") as f:
        pickle.dump(payload, f)


def export_snapshot(params, cfg: Config, dtype, out_dir: Path) -> None:
    xs = jnp.linspace(0.0, 1.0, cfg.export_grid_nx, dtype=dtype)
    ys = jnp.linspace(0.0, 1.0, cfg.export_grid_ny, dtype=dtype)
    X, Y = jnp.meshgrid(xs, ys, indexing="ij")
    Tm = jnp.full_like(X, cfg.export_time)
    points = jnp.stack([X.reshape(-1), Y.reshape(-1), Tm.reshape(-1)], axis=1)
    out = forward_all(params, points, cfg)
    is_fuel = points[:, 0] <= cfg.x_fuel_max + 1.0e-12
    phi_piece = jnp.where(is_fuel, out[:, 0], jnp.nan)
    T_piece = jnp.where(is_fuel, out[:, 1], out[:, 2])
    arr = jnp.stack([points[:, 0], points[:, 1], points[:, 2], phi_piece, T_piece], axis=1)

    path = out_dir / f"snapshot_t_{cfg.export_time:g}.csv"
    rows = jax.device_get(arr).tolist()
    with path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["x", "y", "t", "phi_fuel_only", "T_piecewise"])
        writer.writerows(rows)
    print(f"Saved snapshot: {path}")


# =============================================================================
# Training driver
# =============================================================================

def train(cfg: Config):
    dtype = dtype_from_config(cfg)
    device = choose_device(cfg.device)

    out_dir = Path(cfg.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    write_config(cfg, out_dir)
    loss_csv = out_dir / "loss_history.csv"
    write_loss_header(loss_csv)

    print("JAX version:", jax.__version__)
    print("Using device:", device)
    print("Using dtype:", cfg.dtype)
    print("JIT:", cfg.use_jit)
    print("Fresh resampling every Adam step: yes")
    print("Added jitter/noise: no")
    print("Output directory:", out_dir.resolve())
    print("Point counts per Adam step:")
    print(f"  PDE: phi={cfg.n_phi_pde}, Ts={cfg.n_ts_pde}, Tf={cfg.n_tf_pde}")
    print(f"  phi BC: left={cfg.n_phi_left}, bottom={cfg.n_phi_bottom}, top={cfg.n_phi_top}, interface_N={cfg.n_phi_interface_neumann}")
    print(f"  Ts BC: left={cfg.n_ts_left}, bottom={cfg.n_ts_bottom}, top={cfg.n_ts_top}")
    print(f"  Tf BC: bottom={cfg.n_tf_bottom}, top={cfg.n_tf_top}, right={cfg.n_tf_right}")
    print(f"  IC: phi={cfg.n_ic_phi}, Ts={cfg.n_ic_ts}, Tf={cfg.n_ic_tf}")
    print(f"  Interface: {cfg.n_interface}")

    with jax.default_device(device):
        key = random.PRNGKey(cfg.seed)
        key, k_init = random.split(key)
        params = init_params(k_init, cfg, dtype)
        opt_state = init_adam_state(params, dtype)
        params = jax.device_put(params, device)
        opt_state = jax.device_put(opt_state, device)
        train_step = make_train_step(cfg, dtype)

        start_time = time.time()
        for step in range(1, cfg.steps + 1):
            key, k_step = random.split(key)
            k_step = jax.device_put(k_step, device)
            params, opt_state, logs = train_step(params, opt_state, k_step, jnp.asarray(step, dtype=jnp.int32))

            if step == 1 or step % cfg.log_every == 0:
                elapsed = time.time() - start_time
                append_loss_row(loss_csv, step, elapsed, logs)
                print(
                    f"step {step:7d} | "
                    f"lr={safe_float(logs['lr']):.3e} | "
                    f"total={safe_float(logs['total']):.4e} | "
                    f"phi_pde={safe_float(logs['phi_pde']):.2e} | "
                    f"Ts_pde={safe_float(logs['ts_pde']):.2e} | "
                    f"Tf_pde={safe_float(logs['tf_pde']):.2e} | "
                    f"I_T={safe_float(logs['interface_T']):.2e} | "
                    f"I_q={safe_float(logs['interface_flux']):.2e} | "
                    f"grad={safe_float(logs['grad_norm']):.2e}"
                )

            if cfg.save_every > 0 and step % cfg.save_every == 0:
                save_checkpoint(params, opt_state, cfg, step, out_dir)

        save_checkpoint(params, opt_state, cfg, cfg.steps, out_dir)
        export_snapshot(params, cfg, dtype, out_dir)

    return params, out_dir


# =============================================================================
# CLI
# =============================================================================

def parse_args(argv=None) -> Config:
    p = argparse.ArgumentParser(description="JAX Adam PINN for MOOSE neutron--thermal benchmark")

    p.add_argument("--steps", type=int, default=Config.steps)
    p.add_argument("--lr", type=float, default=Config.lr)
    p.add_argument("--lr-final", type=float, default=Config.lr_final)
    p.add_argument("--lr-schedule", type=str, default=Config.lr_schedule, choices=["constant", "cosine"])
    p.add_argument("--lr-warmup-steps", type=int, default=Config.lr_warmup_steps)
    p.add_argument("--device", type=str, default=Config.device)
    p.add_argument("--dtype", type=str, default=Config.dtype, choices=["float32", "float64"])
    p.add_argument("--seed", type=int, default=Config.seed)
    p.add_argument("--jit", dest="use_jit", action="store_true", default=Config.use_jit)
    p.add_argument("--no-jit", dest="use_jit", action="store_false")

    p.add_argument("--width", type=int, default=Config.width)
    p.add_argument("--layers", type=int, default=Config.layers)
    p.add_argument("--activation", type=str, default=Config.activation, choices=["tanh", "silu"])

    # Point counts. All are freshly resampled every Adam step.
    p.add_argument("--n-phi-pde", type=int, default=Config.n_phi_pde)
    p.add_argument("--n-ts-pde", "--n-Ts-pde", dest="n_ts_pde", type=int, default=Config.n_ts_pde)
    p.add_argument("--n-tf-pde", "--n-Tf-pde", dest="n_tf_pde", type=int, default=Config.n_tf_pde)

    p.add_argument("--n-phi-left", type=int, default=Config.n_phi_left)
    p.add_argument("--n-phi-bottom", type=int, default=Config.n_phi_bottom)
    p.add_argument("--n-phi-top", type=int, default=Config.n_phi_top)
    p.add_argument("--n-phi-interface-neumann", type=int, default=Config.n_phi_interface_neumann)

    p.add_argument("--n-ts-left", type=int, default=Config.n_ts_left)
    p.add_argument("--n-ts-bottom", type=int, default=Config.n_ts_bottom)
    p.add_argument("--n-ts-top", type=int, default=Config.n_ts_top)

    p.add_argument("--n-tf-bottom", type=int, default=Config.n_tf_bottom)
    p.add_argument("--n-tf-top", type=int, default=Config.n_tf_top)
    p.add_argument("--n-tf-right", type=int, default=Config.n_tf_right)

    p.add_argument("--n-ic-phi", type=int, default=Config.n_ic_phi)
    p.add_argument("--n-ic-ts", type=int, default=Config.n_ic_ts)
    p.add_argument("--n-ic-tf", type=int, default=Config.n_ic_tf)
    p.add_argument("--n-interface", type=int, default=Config.n_interface)

    # Weights
    p.add_argument("--w-phi-pde", type=float, default=Config.w_phi_pde)
    p.add_argument("--w-ts-pde", type=float, default=Config.w_ts_pde)
    p.add_argument("--w-tf-pde", type=float, default=Config.w_tf_pde)
    p.add_argument("--w-phi-left", type=float, default=Config.w_phi_left)
    p.add_argument("--w-phi-neumann", type=float, default=Config.w_phi_neumann)
    p.add_argument("--w-ts-neumann", type=float, default=Config.w_ts_neumann)
    p.add_argument("--w-tf-bottom", type=float, default=Config.w_tf_bottom)
    p.add_argument("--w-tf-neumann", type=float, default=Config.w_tf_neumann)
    p.add_argument("--w-ic-phi", type=float, default=Config.w_ic_phi)
    p.add_argument("--w-ic-ts", type=float, default=Config.w_ic_ts)
    p.add_argument("--w-ic-tf", type=float, default=Config.w_ic_tf)
    p.add_argument("--w-interface-T", type=float, default=Config.w_interface_T)
    p.add_argument("--w-interface-flux", type=float, default=Config.w_interface_flux)

    # Group schedules
    p.add_argument("--weight-schedule", type=str, default=Config.weight_schedule, choices=["none", "linear", "cosine"])
    p.add_argument("--weight-warmup-steps", type=int, default=Config.weight_warmup_steps)
    p.add_argument("--pde-mult-start", type=float, default=Config.pde_mult_start)
    p.add_argument("--pde-mult-end", type=float, default=Config.pde_mult_end)
    p.add_argument("--bc-mult-start", type=float, default=Config.bc_mult_start)
    p.add_argument("--bc-mult-end", type=float, default=Config.bc_mult_end)
    p.add_argument("--ic-mult-start", type=float, default=Config.ic_mult_start)
    p.add_argument("--ic-mult-end", type=float, default=Config.ic_mult_end)
    p.add_argument("--interface-mult-start", type=float, default=Config.interface_mult_start)
    p.add_argument("--interface-mult-end", type=float, default=Config.interface_mult_end)

    p.add_argument("--grad-clip-norm", type=float, default=Config.grad_clip_norm)
    p.add_argument("--out-dir", type=str, default=Config.out_dir)
    p.add_argument("--log-every", type=int, default=Config.log_every)
    p.add_argument("--save-every", type=int, default=Config.save_every)
    p.add_argument("--export-grid-nx", type=int, default=Config.export_grid_nx)
    p.add_argument("--export-grid-ny", type=int, default=Config.export_grid_ny)
    p.add_argument("--export-time", type=float, default=Config.export_time)

    # Notebook-safe: ignore Jupyter/IPython unknowns like --f=kernel.json.
    args, _unknown = p.parse_known_args(argv)

    cfg = Config()
    cfg.steps = args.steps
    cfg.lr = args.lr
    cfg.lr_final = args.lr_final
    cfg.lr_schedule = args.lr_schedule
    cfg.lr_warmup_steps = args.lr_warmup_steps
    cfg.device = args.device
    cfg.dtype = args.dtype
    cfg.seed = args.seed
    cfg.use_jit = args.use_jit

    cfg.width = args.width
    cfg.layers = args.layers
    cfg.activation = args.activation

    cfg.n_phi_pde = args.n_phi_pde
    cfg.n_ts_pde = args.n_ts_pde
    cfg.n_tf_pde = args.n_tf_pde
    cfg.n_phi_left = args.n_phi_left
    cfg.n_phi_bottom = args.n_phi_bottom
    cfg.n_phi_top = args.n_phi_top
    cfg.n_phi_interface_neumann = args.n_phi_interface_neumann
    cfg.n_ts_left = args.n_ts_left
    cfg.n_ts_bottom = args.n_ts_bottom
    cfg.n_ts_top = args.n_ts_top
    cfg.n_tf_bottom = args.n_tf_bottom
    cfg.n_tf_top = args.n_tf_top
    cfg.n_tf_right = args.n_tf_right
    cfg.n_ic_phi = args.n_ic_phi
    cfg.n_ic_ts = args.n_ic_ts
    cfg.n_ic_tf = args.n_ic_tf
    cfg.n_interface = args.n_interface

    cfg.w_phi_pde = args.w_phi_pde
    cfg.w_ts_pde = args.w_ts_pde
    cfg.w_tf_pde = args.w_tf_pde
    cfg.w_phi_left = args.w_phi_left
    cfg.w_phi_neumann = args.w_phi_neumann
    cfg.w_ts_neumann = args.w_ts_neumann
    cfg.w_tf_bottom = args.w_tf_bottom
    cfg.w_tf_neumann = args.w_tf_neumann
    cfg.w_ic_phi = args.w_ic_phi
    cfg.w_ic_ts = args.w_ic_ts
    cfg.w_ic_tf = args.w_ic_tf
    cfg.w_interface_T = args.w_interface_T
    cfg.w_interface_flux = args.w_interface_flux

    cfg.weight_schedule = args.weight_schedule
    cfg.weight_warmup_steps = args.weight_warmup_steps
    cfg.pde_mult_start = args.pde_mult_start
    cfg.pde_mult_end = args.pde_mult_end
    cfg.bc_mult_start = args.bc_mult_start
    cfg.bc_mult_end = args.bc_mult_end
    cfg.ic_mult_start = args.ic_mult_start
    cfg.ic_mult_end = args.ic_mult_end
    cfg.interface_mult_start = args.interface_mult_start
    cfg.interface_mult_end = args.interface_mult_end

    cfg.grad_clip_norm = args.grad_clip_norm
    cfg.out_dir = args.out_dir
    cfg.log_every = args.log_every
    cfg.save_every = args.save_every
    cfg.export_grid_nx = args.export_grid_nx
    cfg.export_grid_ny = args.export_grid_ny
    cfg.export_time = args.export_time
    return cfg


if __name__ == "__main__":
    train(parse_args())


JAX version: 0.6.2
Using device: cuda:0
Using dtype: float32
JIT: True
Fresh resampling every Adam step: yes
Added jitter/noise: no
Output directory: /projects/549120ce-e7e0-45e0-a453-c9903649fea2/sqp/adam_jax_moose_output
Point counts per Adam step:
  PDE: phi=512, Ts=512, Tf=768
  phi BC: left=128, bottom=128, top=128, interface_N=128
  Ts BC: left=128, bottom=128, top=128
  Tf BC: bottom=128, top=128, right=128
  IC: phi=128, Ts=128, Tf=128
  Interface: 256


2026-07-12 10:30:20.687192: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-07-12 10:30:20.687238: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-07-12 10:30:20.687291: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-07-12 10:30:20.687340: W external/xla/xla/service/gpu/au

step       1 | lr=1.000e-03 | total=8.2917e+01 | phi_pde=7.32e-02 | Ts_pde=5.06e+00 | Tf_pde=1.93e+00 | I_T=3.91e-01 | I_q=1.24e+00 | grad=8.44e+02
step     100 | lr=1.000e-03 | total=3.5454e+00 | phi_pde=6.26e-01 | Ts_pde=5.84e+00 | Tf_pde=7.03e-02 | I_T=9.80e-03 | I_q=4.16e-02 | grad=8.32e+01
step     200 | lr=1.000e-03 | total=1.6559e+00 | phi_pde=7.38e-01 | Ts_pde=1.11e+00 | Tf_pde=2.96e-02 | I_T=4.53e-03 | I_q=3.32e-03 | grad=8.41e+01
step     300 | lr=9.999e-04 | total=8.2380e-01 | phi_pde=7.07e-01 | Ts_pde=2.55e-01 | Tf_pde=2.53e-02 | I_T=1.66e-03 | I_q=1.58e-03 | grad=6.46e+01
step     400 | lr=9.999e-04 | total=8.9132e-01 | phi_pde=6.95e-01 | Ts_pde=2.02e-01 | Tf_pde=9.35e-03 | I_T=3.52e-03 | I_q=2.79e-03 | grad=8.23e+01
step     500 | lr=9.998e-04 | total=8.4348e-01 | phi_pde=6.89e-01 | Ts_pde=1.08e-01 | Tf_pde=9.76e-03 | I_T=3.22e-03 | I_q=2.14e-03 | grad=6.77e+01
step     600 | lr=9.997e-04 | total=9.7062e-01 | phi_pde=5.95e-01 | Ts_pde=2.79e-01 | Tf_pde=1.27e-02 | I_T=3.35

phi_true.shape = (17, 20, 64)
axis sizes: 17 20 64
axis1 first  phi_true[0,:,:]
  shape   : (20, 64)
  mean abs: 1.2736294023845356
  min/max : 0.05063449870848282 1.998796676955403
axis1 last   phi_true[-1,:,:]
  shape   : (20, 64)
  mean abs: 2.7600419738549364
  min/max : 0.04744169422884767 10.483047280042607
axis2 first  phi_true[:,0,:]
  shape   : (17, 64)
  mean abs: 3.487378040282388
  min/max : 0.05063449870848282 10.483047280042607
axis2 last   phi_true[:,-1,:]
  shape   : (17, 64)
  mean abs: 0.1759052020626532
  min/max : 0.019286873502372324 1.998796676955403
axis3 first  phi_true[:,:,0]
  shape   : (17, 20)
  mean abs: 0.8020025868223284
  min/max : 0.05063449870848282 1.6646823997930156
axis3 last   phi_true[:,:,-1]
  shape   : (17, 20)
  mean abs: 0.5493111143768257
  min/max : 0.05063449870848349 0.8793695867465514
